<a href="https://colab.research.google.com/github/Ranis1337/multimodal-image-to-speech/blob/main/%D0%98%D0%98%D0%9D%D0%BC_1_26_%D0%A8%D0%B0%D0%BC%D0%B8%D0%BB%D0%BE%D0%B2%D0%A0%D0%9A_%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%BD%D1%8B%D0%B9_%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D1%83%D0%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gradio as gr
from PIL import Image
import scipy.io.wavfile
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BlipForConditionalGeneration,
    BlipProcessor,
    VitsModel,
    VitsTokenizer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

print("1/3 Загрузка BLIP...")
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

print("2/3 Загрузка переводчика...")
trans_model_name = "Helsinki-NLP/opus-mt-en-ru"
trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

print("3/3 Загрузка модели синтеза речи...")
tts_tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-rus")
tts_model = VitsModel.from_pretrained("facebook/mms-tts-rus").to(device)
tts_sampling_rate = tts_model.config.sampling_rate

def translate_text(text: str) -> str:
  inputs = trans_tokenizer(text, return_tensors="pt", padding=True).to(device)
  with torch.no_grad():
    translated_tokens = trans_model.generate(**inputs)
  return trans_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

def process_image_pipeline(image: Image.Image):
  if image is None:
    return "Загрузите изображение", "—", None

  inputs = blip_processor(image, return_tensors="pt").to(device)
  with torch.no_grad():
    out = blip_model.generate(**inputs, max_new_tokens=50)
  en_caption = blip_processor.decode(out[0], skip_special_tokens=True)

  ru_caption = translate_text(en_caption)

  tts_inputs = tts_tokenizer(ru_caption, return_tensors="pt").to(device)
  with torch.no_grad():
    audio_output = tts_model(**tts_inputs).waveform


  audio_data = audio_output.squeeze().cpu().numpy()

  return en_caption, ru_caption, (tts_sampling_rate, audio_data)

demo = gr.Interface(
    fn=process_image_pipeline,
    inputs=gr.Image(type="pil", label="Исходное изображение"),
    outputs=[
        gr.Textbox(label="1. Описание на английском (BLIP)"),
        gr.Textbox(label="2. Перевод на русский (MarianMT)"),
        gr.Audio(label="3. Озвучка на русском (MMS-TTS)"),
    ],
    title="Мультимодальный ассистент: «Глаза + Редактор + Голос»",
    description="Пайплайн из 3 моделей: Vision-to-Text (BLIP) -> Перевод (MarianMT) -> Озвучка (Facebook MMS-TTS).",
)

demo.launch(share=True)

Используемое устройство: cuda
1/3 Загрузка BLIP...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a1efa6907e0ef1b9f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
